# CSC725 Homework 2 - step-by-step walkthrough
This notebook uses `hw2_solution.py` as the single implementation source.
The supplied experiment was actually run through that script; its measured
results are in `results/`. These walkthrough cells have not been executed here.
By default, they load the completed experiment. Set `RETRAIN=True` below to train
a new run. Keep the notebook and Python file together.


In [ ]:
from pathlib import Path
import json
import torch
from torch import nn
from IPython.display import display, Image
from hw2_solution import (ThreeLayerCNN, TrainConfig, choose_device, make_loaders,
                         count_parameters, calculate_flops, run_training)
ROOT = Path.cwd()
config = TrainConfig(data_dir=str(ROOT / "data"), output_dir=str(ROOT / "results_notebook"))


## 1. Data loading
CIFAR-10 has 50,000 training images and 10,000 test images. The transform maps
RGB values to [-1,1]. Only the training loader shuffles its examples.


In [ ]:
train_loader, test_loader = make_loaders(config)
assert len(train_loader.dataset) == 50_000
assert len(test_loader.dataset) == 10_000
images, labels = next(iter(train_loader))
print("Batch shape:", images.shape, "Label shape:", labels.shape)


## 2. Model and shapes
Each convolution preserves spatial size; each pool halves it. The final
64x4x4 tensor gives 1,024 classifier inputs and ten raw output logits.


In [ ]:
model = ThreeLayerCNN()
print(model)
assert model(images[:2]).shape == (2, 10)
print("Classifier input width:", model.fc.in_features)


## 3. Bonus calculations, including independent hand checks
The assignment counts multiplications only. ReLU, pooling, bias additions, and
all other additions are excluded. The functions implement the formulas directly;
there is no profiler dependency.


In [ ]:
parameter_results = count_parameters(model)
flop_results = calculate_flops(model, (1, 3, 32, 32))
assert parameter_results["total"] == 16*(3*3*3+1) + 32*(16*3*3+1) + 64*(32*3*3+1) + 10*(1024+1)
assert flop_results["total_flops"] == 32*32*16*3*9 + 16*16*32*16*9 + 8*8*64*32*9 + 1024*10
assert parameter_results["total"] == 33_834
assert flop_results["total_flops"] == 2_811_904


## 4. Full training and final testing
The script uses CrossEntropyLoss, Adam, shuffled batches, and at least ten full
epochs. It records loss and accuracy after every epoch. The final model is tested
after training using eval mode and no gradient tracking. It saves all predictions.
The default cell reads the already completed run so it does not repeat training.


In [ ]:
RETRAIN = False
if RETRAIN:
    experiment = run_training(config)  # choose a fresh output directory if re-running
    result_dir = Path(config.output_dir)
else:
    result_dir = ROOT / "results"
    experiment = json.loads((result_dir / "results.json").read_text())
print("Epoch  Loss       Accuracy")
for row in experiment["history"]:
    print(f'{row["epoch"]:>5}  {row["loss"]:.6f}   {100*row["accuracy"]:.2f}%')
print("Final test accuracy:", f'{100*experiment["test"]["accuracy"]:.2f}%')
print("Test examples:", experiment["test"]["samples"])


## 5. Required training graphs

In [ ]:
display(Image(filename=str(result_dir / "training_loss.png")))
display(Image(filename=str(result_dir / "training_accuracy.png")))


## 6. Verify saved results and checkpoint
The verifier checks sample counts, logs, predictions, complexity, source
fingerprint, and checkpoint loading. `reload_test=True` additionally evaluates
the saved model on the full test split and compares every prediction.


In [ ]:
from verify_run import verify
verification = verify(result_dir, reload_test=True)
print(json.dumps(verification, indent=2))


## 7. Report and understanding
Read `HW2_Report.pdf` for the measured experiment and complete bonus answers.
Use `README.md` for the derivations and training-loop explanation. The model
needs 33,834 learnable parameters and 2,811,904 counted multiplications/image.
Lower training loss alone does not establish good performance on unseen images;
that is why the separate test split is evaluated.
